In [ ]:

import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, 'HI-SIGMA')
from utils import *
from mrl_abstract_functions import *

# HI-SIGMA

We have created a dataset following HI-SIGMA, and trained a signal and a background model.

Directory files

In [ ]:
data_dir = "HI-SIGMA/datasets/4feats_smear2/"
results_dir = "results_HI_SIGMA_with_syst_with_mask/"
background_dir = "HI-SIGMA/bkg_models_new/4feats_smear2"
signal_dir ="HI-SIGMA/sig_models_new/4feats_smear2"
os.makedirs(results_dir, exist_ok=True)

HI-SIGMA hyperparameters

In [ ]:
zero_align_profiles = True
SB_flow_mask = True
synthetic_signal = False

sample_size = 50000

feat_names = [
    r"$\Delta R_{\gamma\gamma}$",
    r"$\Delta R_{bb}$",
    r"H$_{\gamma\gamma}$ $p_T / M$",
    r"H$_{bb}$ $p_T$",
    "Total Mass",
    r"$\Delta R_{\gamma^1 b^1}$",
]

bkg_scaler = joblib.load(data_dir + "bkg_scaler.pkl")
sig_scaler = joblib.load(data_dir + "sig_scaler.pkl")


# flow params
bkg_flow_num_layers = 6
bkg_flow_hidden_size = 64
bkg_flow_num_bins = 24
RQS = True

sig_flow_num_layers = 6
sig_flow_hidden_size = 64
sig_flow_num_bins = 24

Load data

In [ ]:
device = "cpu"

# data_holdout_full = np.load(data_dir + "data.npy")
# labels_holdout_full = np.load(data_dir + "labels.npy")

data_holdout_full = np.load(data_dir + "data_holdout.npy")
labels_holdout_full = np.load(data_dir + "labels_holdout.npy")

# for fitting signal shape
sim = np.load(data_dir + "sim.npy")

nfeatures = data_holdout_full.shape[1] - 1

mass_low_sr = convert_mgg(115.0)
mass_high_sr = convert_mgg(135.0)

mass_low_mask_sr = convert_mgg(105.0)
mass_high_mask_sr = convert_mgg(145.0)

cut_low = convert_mgg(90.0)
cut_high = convert_mgg(180.0)

Define number of available datasets

In [ ]:
data_iter_max = int(len(data_holdout_full)/sample_size)+1
print(data_iter_max,len(data_holdout_full))
for i in range(data_iter_max):
    print(i,i * sample_size,(i+1) * sample_size)
    if i < data_iter_max -1:
        print((i+1) * sample_size,(i+2) * sample_size)
    else:
        print(0,sample_size)

## Shared functions

We assume the crystal ball is always the same across pseudo-experiments

In [ ]:
# Fit signal crystall ball shape
sig_masses = sim[:, 0:1]
sig_masses = sig_masses[:200000]
sig_masses_mask = (sig_masses > mass_low_sr + 0.01) & (sig_masses < mass_high_sr - 0.01)
sig_masses_fit = sig_masses[sig_masses_mask]
c = cost.UnbinnedNLL(sig_masses_fit, pdf_DCB)


loc = np.mean(sig_masses_fit)
m_sig_dcb = Minuit(
    c,
    m_left=2.0,
    beta_left=2.0,
    scale_left=0.5,
    beta_right=2.0,
    m_right=2.0,
    scale_right=0.5,
    loc=loc,
)
m_sig_dcb.limits["scale_left"] = (0, None)
m_sig_dcb.limits["scale_right"] = (0, None)
m_sig_dcb.limits["beta_left"] = (1, None)
m_sig_dcb.limits["beta_right"] = (1, None)

m_sig_dcb.migrad()  # finds minimum
m_sig_dcb.hesse()  # accurately computes uncertainties

joblib.dump(m_sig_dcb.values[:], results_dir + "dcb_fit.pkl")


(
    beta_left_fit,
    m_left_fit,
    scale_left_fit,
    beta_right_fit,
    m_right_fit,
    scale_right_fit,
    loc_fit,
) = m_sig_dcb.values



And the same goes for the signal and background models, trained in simulation and sidebands, respectively

In [ ]:
model_path = os.path.join(
    background_dir, "background_model.par"
)
flowB = flow_model(
    nfeatures,
    num_layers=bkg_flow_num_layers,
    hidden_size=bkg_flow_hidden_size,
    RQS=RQS,
    num_bins=bkg_flow_num_bins,
)
flowB.load_state_dict(torch.load(model_path, weights_only=True))
flowB.to(device)
flowB.eval()

In [ ]:
model_path = os.path.join(
    signal_dir, "signal_model.par" 
)
flowS = flow_model(
    nfeatures,
    num_layers=sig_flow_num_layers,
    hidden_size=sig_flow_hidden_size,
    RQS=RQS,
    num_bins=sig_flow_num_bins,
)
flowS.load_state_dict(torch.load(model_path, weights_only=True))
flowS.to(device)
flowS.eval()

We define the functions that are needed

In [ ]:
samples_per_model = 1
def ensemble_sample(models, context, scaler=None, num_samples=10):
    samples = []
    for model in models:
        model.to("cpu")
        sample = model.sample(num_samples=num_samples, context=context).detach().numpy()
        sample = sample.reshape(-1, sample.shape[-1])
        samples.append(sample)

    samples = np.squeeze(np.concatenate(samples, axis=0))

    if scaler is not None:
        samples = scaler.inverse_transform(samples)

    return samples

def temp_cost_func(bkg_norm, sig_norm):
    exp = bkg_norm * bkg_temp + sig_norm * sig_temp
    return poisson_chi2(data_temp, exp)


# define different likelihood functions for various fits


def pdf_bkg_only_m(x, s, p0, p1, p2):
    bkg_shape_norm = fit_fn.norm([p0, p1, p2])
    norm = bkg_shape_norm
    bkg_mass_prob = fit_fn.density(x, [p0, p1, p2])


    L = bkg_mass_prob
    return norm, L

def pdf_comb_only_m(x, s, p0, p1, p2):
    bkg_shape_norm = fit_fn.norm([p0, p1, p2])
    norm = s + bkg_shape_norm
    bkg_mass_prob = fit_fn.density(x, [p0, p1, p2])
    sig_mass_prob = cb.pdf(
        x,
        beta_left_fit,
        m_left_fit,
        scale_left_fit,
        beta_right_fit,
        m_right_fit,
        scale_right_fit,
        loc_fit,
    )

    L = bkg_mass_prob + s * sig_mass_prob
    return norm, L

def pdf_comb_full(x, s, p0, p1, p2):
    bkg_shape_norm = fit_fn.norm([p0, p1, p2])
    norm = s + bkg_shape_norm
    bkg_mass_prob = fit_fn.density(x, [p0, p1, p2])
    sig_mass_prob = cb.pdf(
        x,
        beta_left_fit,
        m_left_fit,
        scale_left_fit,
        beta_right_fit,
        m_right_fit,
        scale_right_fit,
        loc_fit,
    )

    L = bkg_mass_prob * bkg_flow_prob + s * sig_mass_prob * sig_flow_prob
    return norm, L

def analyser(
    func, masses=None, verbose=True, grid=None, s_guess=0, syst=False, plt_name=""
):

    norm = len(masses)

    c_comb = cost.ExtendedUnbinnedNLL(masses, func)
    m = Minuit(c_comb, s=s_guess, p0=norm, p1=p1_guess, p2=p2_guess)

    m.limits["p0"] = p0_range
    m.limits["p1"] = p1_range
    m.limits["p2"] = p2_range
    m.fixed["s"] = False
    # m.fixed['alpha']=True

    m.migrad()  # finds minimum
    m.minos()#m.hesse()

    if grid is None:
        num_grid = 60
        grid = np.linspace(
            m.values["s"] - 3 * m.errors["s"],
            m.values["s"] + 3 * m.errors["s"],
            num_grid,
        )
        s_values, s_profile_likelihood, s_convergence = m.mnprofile("s", grid=grid)
    else:
        s_values, s_profile_likelihood, s_convergence = m.mnprofile("s", grid=grid)

    eps = 1e-8
    best_s = m.values["s"]
    best_s_err = m.errors["s"] + eps

    L_s_free = func(masses, *m.values)[1]
    NLL_s_free = -np.sum(np.log(L_s_free))
    # print(NLL_s_free)

    if verbose:
        print(
            "best fit signal strength is %.2f +/- %.2f. Naive signif %.2f"
            % (best_s, best_s_err, best_s / best_s_err)
        )
        print("NLL %.2f" % NLL_s_free)

    c_comb = cost.ExtendedUnbinnedNLL(masses, func)
    m_fixed_s = Minuit(c_comb, s=0, p0=norm, p1=p1_guess, p2=p2_guess)

    m_fixed_s.limits["p0"] = p0_range
    m_fixed_s.limits["p1"] = p1_range
    m_fixed_s.limits["p2"] = p2_range
    m_fixed_s.fixed["s"] = True
    # m_fixed_s.fixed['alpha']=True

    if len(plt_name) > 0:
        print("drawing fit %s" % plt_name)
        draw_mass_postfit(masses, m.values, fname=plt_name)

    m_fixed_s.migrad()  # finds minimum
    m_fixed_s.hesse()  # accurately computes uncertainties

    L_s_fixed = func(masses, *m_fixed_s.values)[1]
    NLL_s_fixed = -np.sum(np.log(L_s_fixed))

    delta_NLL = 2.0 * (NLL_s_fixed - NLL_s_free)
    # avoid errors
    delta_NLL = max(0.0, delta_NLL)

    if verbose:
        print(
            "Delta NLL is %.1f. Approx significance is %.2f"
            % (delta_NLL, delta_NLL**0.5)
        )
        print(m.values)

    out = {
        "s_values": s_values,
        "s_profile": s_profile_likelihood,
        "best_s": best_s,
        "best_s_err": best_s_err,
        "NLL_s_free": NLL_s_free,
        "NLL_s_fixed": NLL_s_fixed,
        "delta_NLL": delta_NLL,
        "params": m.values[:],
    }

    return out

def analyser_no_signal(
    func, masses=None, verbose=True, grid=None, s_guess=0, syst=False, plt_name=""
):

    norm = len(masses)

    c_comb = cost.ExtendedUnbinnedNLL(masses, func)
    m = Minuit(c_comb, s=s_guess, p0=norm, p1=p1_guess, p2=p2_guess)

    m.limits["p0"] = p0_range
    m.limits["p1"] = p1_range
    m.limits["p2"] = p2_range
    m.fixed["s"] = True
    # m.fixed['alpha']=True

    m.migrad()  # finds minimum
    m.minos()#m.hesse()

    if verbose:

        print(m.values)
        print(m.errors)

    out = {
        "params": m.values[:],
        "errors": m.errors[:],
    }

    return out
#'''
def pdf_comb_full_Lmin(x, s, b, p0, p1, p2):
    ### change to use calibration data for eff_sr computation

    # norm = s + b
    # bkg_mass_norm = fit_fn.norm([p0, p1, p2])
    bkg_mass_prob = fit_fn.pdf(x, [p0, p1, p2])
    # bkg_mass_prob_nom = fit_fn.pdf(x, [p0_fit, p1_fit, p2_fit])
    # weights = bkg_mass_prob / bkg_mass_prob_nom

    bkg_mass_prob_toy = fit_fn.pdf(data_masses_toys[all_prob_toys>Lmin], [p0, p1, p2])
    bkg_mass_prob_nom_toy = fit_fn.pdf(data_masses_toys[all_prob_toys>Lmin], [p0_fit, p1_fit, p2_fit])
    weights_toy = np.mean(bkg_mass_prob_toy/bkg_mass_prob_nom_toy)
    eff_toy_sr = eff_toy * np.mean(weights_toy)
    
    bkg_mass_prob_cal = fit_fn.pdf(data_masses_cal[all_prob_cal>Lmin], [p0, p1, p2])
    bkg_mass_prob_nom_cal = fit_fn.pdf(data_masses_cal[all_prob_cal>Lmin], [p0_fit, p1_fit, p2_fit])
    weights_cal = np.mean(bkg_mass_prob_cal/bkg_mass_prob_nom_cal)
    eff_cal_sr = eff_cal * np.mean(weights_cal)
    # norm = s + bkg_mass_norm * eff_sr

    norm = s * eff_sim + b * eff_cal_sr

    sig_mass_prob = cb.pdf(
        x,
        beta_left_fit,
        m_left_fit,
        scale_left_fit,
        beta_right_fit,
        m_right_fit,
        scale_right_fit,
        loc_fit,
    )
    # print(eff_cal,eff_toy, eff_sr)
    # print(sig_mass_prob.shape)

    L = (
         b * (bkg_mass_prob)  * (bkg_flow_prob[mask_mrl])* eff_cal_sr / eff_toy_sr
        # bkg_mass_norm * eff_sr * (bkg_mass_prob) * (bkg_flow_prob[all_prob_flow>Lmin]) / eff_sr
        + s * sig_mass_prob * (sig_flow_prob[mask_mrl])
    )    
    if np.sum(L<0)>0:
        print("Negative events")
        print(np.sum(L<0))
        print(np.where(L<0),L[L<0])
        print(s,b,bkg_mass_prob[L<0],sig_mass_prob[L<0],bkg_flow_prob[mask_mrl][L<0],sig_flow_prob[mask_mrl][L<0])
        # L=np.where(L<0,L,b *(bkg_mass_prob) * (bkg_flow_prob[all_prob_flow>Lmin]) * eff_cal_sr / eff_toy_sr)
    return norm, L

def analyser_full_Lmin(
    func, masses=None, verbose=True, grid=None, s_guess=0, syst=False, plt_name=""
):

    norm = len(masses)

    c_comb = cost.ExtendedUnbinnedNLL(masses, func)

    try:

        # m = Minuit(c_comb, s=s_guess, b=norm)
        # c_comb += cost.NormalConstraint(["p0", "p1", "p2"], [p0_fit, p1_fit, p2_fit], [err_p0_fit, err_p1_fit, err_p2_fit])
        c_comb += cost.NormalConstraint("p0", p0_fit, err_p0_fit)
        c_comb += cost.NormalConstraint("p1", p1_fit, err_p1_fit)
        c_comb += cost.NormalConstraint("p2", p2_fit, err_p2_fit) 
        m = Minuit(c_comb, s=s_guess, p0=p0_fit, p1=p1_fit, p2=p2_fit,b=norm)

        # m.limits["b"] = (0,norm)
        m.limits["p0"] = p0_range
        m.limits["p1"] = p1_range
        m.limits["p2"] = p2_range        
        m.fixed["s"] = False
        # m.limits["s"] = (0,None)
        # m.fixed["p0"] = True
        # m.fixed["p1"] = True
        # m.fixed["p2"] = True                        
        # m.fixed['alpha']=True

        m.migrad()  # finds minimum
        m.minos()#m.hesse()
    except:
        
        c_comb = cost.ExtendedUnbinnedNLL(masses, func)
        m = Minuit(c_comb, s=s_guess)

        m.fixed["s"] = False
        # m.fixed['alpha']=True

        m.migrad()  # finds minimum
        m.minos()#m.hesse()

    if grid is None:
        num_grid = 60
        grid = np.linspace(
            m.values["s"] + 3 * float(repr(m.merrors["s"]).split('lower')[1].split('=')[1].split('upper')[0]),
            m.values["s"] + 3 * float(repr(m.merrors["s"]).split('upper')[1].split('=')[1].split('is_valid')[0]),
            num_grid,
        )
        s_values, s_profile_likelihood, s_convergence = m.mnprofile("s", grid=grid)
    else:
        s_values, s_profile_likelihood, s_convergence = m.mnprofile("s", grid=grid)

    eps = 1e-8
    best_s = m.values["s"]
    best_s_err = [eps-float(repr(m.merrors["s"]).split('lower')[1].split('=')[1].split('upper')[0]),eps+float(repr(m.merrors["s"]).split('upper')[1].split('=')[1].split('is_valid')[0])]

    L_s_free = func(masses, *m.values)[1]
    NLL_s_free = -np.sum(np.log(L_s_free))
    # print(NLL_s_free)

    if verbose:
        print(
            "best fit signal strength is %.2f - %.2f + %.2f. Naive signif %.2f"
            % (best_s, best_s_err[0],best_s_err[1], best_s / best_s_err)
        )
        print("NLL %.2f" % NLL_s_free)

    c_comb = cost.ExtendedUnbinnedNLL(masses, func)
    try:
        # m_fixed_s = Minuit(c_comb, s=s_guess, b=norm)
        c_comb += cost.NormalConstraint("p0", p0_fit, err_p0_fit)
        c_comb += cost.NormalConstraint("p1", p1_fit, err_p1_fit)
        c_comb += cost.NormalConstraint("p2", p2_fit, err_p2_fit)   
        m_fixed_s = Minuit(c_comb, s=s_guess, p0=p0_fit, p1=p1_fit, p2=p2_fit, b=norm)

        m_fixed_s.limits["p0"] = p0_range
        m_fixed_s.limits["p1"] = p1_range
        m_fixed_s.limits["p2"] = p2_range    

        # m_fixed_s.limits["b"] = (0,norm)
        m_fixed_s.fixed["s"] = True
        # m_fixed_s.fixed["p0"] = True
        # m_fixed_s.fixed["p1"] = True
        # m_fixed_s.fixed["p2"] = True                        

        # m_fixed_s.fixed['alpha']=True


        m_fixed_s.migrad()  # finds minimum
        m_fixed_s.hesse()  # accurately computes uncertainties
    except:
        c_comb = cost.ExtendedUnbinnedNLL(masses, func)
        m_fixed_s = Minuit(c_comb, s=s_guess)

        m_fixed_s.fixed["s"] = True
        # m_fixed_s.fixed['alpha']=True


        m_fixed_s.migrad()  # finds minimum
        m_fixed_s.hesse()  # accurately computes uncertainties

    L_s_fixed = func(masses, *m_fixed_s.values)[1]
    NLL_s_fixed = -np.sum(np.log(L_s_fixed))

    delta_NLL = 2.0 * (NLL_s_fixed - NLL_s_free)
    # avoid errors
    delta_NLL = max(0.0, delta_NLL)

    if verbose:
        print(
            "Delta NLL is %.1f. Approx significance is %.2f"
            % (delta_NLL, delta_NLL**0.5)
        )
        print(m.values)

    out = {
        "s_values": s_values,
        "s_profile": s_profile_likelihood,
        "best_s": best_s,
        "best_s_err": best_s_err,
        "NLL_s_free": NLL_s_free,
        "NLL_s_fixed": NLL_s_fixed,
        "delta_NLL": delta_NLL,
        "params": m.values[:],
    }

    return out

## Create pseudo-data

In [ ]:
if synthetic_signal == True:
    mass_vals = np.linspace(np.min(sig_masses_fit),np.max(sig_masses_fit),100) 
    Max_Val = 1.01*np.max(cb.pdf(
        mass_vals,
        beta_left_fit,
        m_left_fit,
        scale_left_fit,
        beta_right_fit,
        m_right_fit,
        scale_right_fit,
        loc_fit,))
    # print("Normalization for toy", norm_toy)
    bunch_of_signal_masses = np.zeros(int(160000))
    for i in range(len(bunch_of_signal_masses)):
        while True:
            mass_try = np.random.uniform(np.min(sig_masses_fit),np.max(sig_masses_fit))
            prob_try = cb.pdf(mass_try,beta_left_fit,m_left_fit,scale_left_fit,beta_right_fit,m_right_fit,scale_right_fit,loc_fit,)
            if np.random.uniform(0,Max_Val) < prob_try:
                bunch_of_signal_masses[i] = mass_try
                break

    new_sim = np.zeros((sim.shape[0],nfeatures+1))
    new_sim[:,0] = bunch_of_signal_masses[:new_sim.shape[0]]#cb_scipy.rvs(new_sim.shape[0])


    tmp_samples_new_sim = ensemble_sample(
            [flowS],
            context=torch.tensor(
                new_sim[:,0:1], dtype=torch.float32
            ),
            scaler=sig_scaler,
            num_samples=samples_per_model)


    new_sim[:,1:] = tmp_samples_new_sim

    _,bins,_ = plt.hist(sig_masses_fit,histtype='step',density=True,color='black')
    plt.hist(new_sim[:,0],bins=bins,histtype='step',density=True,color='red')
    plt.show()

    for nfeature in range(nfeatures):
        _, bins, _=plt.hist(sim[:,nfeature+1],histtype='step',color='black')
        plt.hist(new_sim[:,nfeature+1],bins=bins,histtype='step',color='red')
        plt.show()    

    sim = new_sim

    nsignal = np.sum(labels_holdout_full)

    new_sig = np.zeros((nsignal,nfeatures+1))
    new_sig[:,0] = bunch_of_signal_masses[new_sim.shape[0]:new_sim.shape[0]+nsignal]#cb_scipy.rvs(new_sig.shape[0])


    tmp_samples_new_sim = ensemble_sample(
            [flowS],
            context=torch.tensor(
                new_sig[:,0:1], dtype=torch.float32
            ),
            scaler=sig_scaler,
            num_samples=samples_per_model)


    new_sig[:,1:] = tmp_samples_new_sim    

    _,bins,_ = plt.hist(data_holdout_full[labels_holdout_full==1][:,0],histtype='step',density=True,color='black')
    plt.hist(new_sig[:,0],bins=bins,histtype='step',density=True,color='red')
    plt.show()  

    for nfeature in range(nfeatures):
        _, bins, _=plt.hist(data_holdout_full[labels_holdout_full==1][:,nfeature+1],histtype='step',color='black')
        plt.hist(new_sig[:,nfeature+1],bins=bins,histtype='step',color='red')
        plt.show()      

    data_holdout_full[labels_holdout_full==1]=new_sig        

## Simulation and Control Region samples

In [ ]:
sim_feats_cal = sim[:200000,1:]
sim_masses_cal = sim[:200000,0:1].reshape(-1)

sim_feats_cal = sim_feats_cal[(sim_masses_cal > cut_low) & (sim_masses_cal < cut_high)]
sim_masses_cal = sim_masses_cal[(sim_masses_cal > cut_low) & (sim_masses_cal < cut_high)]

# Filter signal to chosen amount

SB_mask_sim = (sim_masses_cal < mass_low_sr) | (sim_masses_cal >= mass_high_sr)
SR_mask_sim = (sim_masses_cal >= mass_low_sr) & (sim_masses_cal < mass_high_sr)

# sim_feats_cal = sim_feats_cal[SB_mask_sim]
# sim_masses_cal = sim_masses_cal[SB_mask_sim]

print("Calibration events", len(sim_feats_cal))

bkg_flow_prob_sim = np.zeros(len(sim_feats_cal))


overall_bkg_probs_sim = []

# separate preprocessing for signal and bkg flows
bkg_inputs_sim = bkg_scaler.transform(sim_feats_cal)

# compute jacobian of preprocessing transformations
bkg_scaler_jacobian = np.ones((len(sim_feats_cal), nfeatures))
for nfeat in range(nfeatures):
    h1 = (
        bkg_scaler["logitscaler"].scale_[nfeat] * sim_feats_cal[:, nfeat]
        + bkg_scaler["logitscaler"].min_[nfeat]
    )
    h1 = np.where(
        h1 < 1 - bkg_scaler["logitscaler"].epsilon,
        h1,
        1 - bkg_scaler["logitscaler"].epsilon,
    )
    h1 = np.where(
        h1 > bkg_scaler["logitscaler"].epsilon, h1, bkg_scaler["logitscaler"].epsilon
    )

    bkg_scaler_jacobian[:, nfeat] = (
        (1 / bkg_scaler["standardscaler"].scale_[nfeat])
        * (1 / (h1 * (1 - h1)))
        * bkg_scaler["logitscaler"].scale_[nfeat]
    )


det_bkg_scaler_jacobian_sim = np.abs(np.prod(bkg_scaler_jacobian, 1))
# print("Bkg jacobian")
# print(np.sum(np.isnan(det_bkg_scaler_jacobian_sim)), np.sum(det_bkg_scaler_jacobian_sim == 0))
# print(det_bkg_scaler_jacobian_sim[:10])

# print("initial Nans")
# print(np.sum(np.isnan(bkg_inputs_sim)))
# cleanup outliers, different procedure for high vs low nans
mean_sim_feats_cal = np.mean(sim_feats_cal, axis=0)
cutoff = 5.0

for i in range(sim_feats_cal.shape[1]):
    above_avg = sim_feats_cal[:, i] > mean_sim_feats_cal[i]
    bkg_inputs_sim[:, i] = clean_nans(bkg_inputs_sim[:, i], above_avg)
bkg_inputs_sim = np.clip(bkg_inputs_sim, -cutoff, cutoff)

bkg_inputs_sim = torch.tensor(bkg_inputs_sim, dtype=torch.float32)

bkg_flow_prob_sim += (
    flowB.log_prob(
        inputs=bkg_inputs_sim,
        context=torch.tensor(sim_masses_cal.reshape(-1, 1), dtype=torch.float32),
    )
    .cpu()
    .detach()
    .numpy()
)

bkg_flow_prob_sim = np.exp(bkg_flow_prob_sim) * det_bkg_scaler_jacobian_sim

if SB_flow_mask:
    bkg_flow_prob_sim[SB_mask_sim] = 1.0 / (2 * cutoff) ** 4

In [ ]:

evt_start_cal = (data_iter_max - 1) * sample_size - 10000
evt_stop_cal = (data_iter_max) * sample_size
# evt_start_cal = 2 * sample_size
# evt_stop_cal = (data_iter_max) * sample_size

# data_iter = data_iter_max-3

# if data_iter < data_iter_max - 1:
#     evt_start_cal = (data_iter + 1) * sample_size
#     evt_stop_cal = (data_iter + 2) * sample_size
# else:
#     evt_start_cal = 0
#     evt_stop_cal = sample_size

# evt_start_cal = (2 - 1) * sample_size
# evt_stop_cal = (data_iter_max) * sample_size


data_feats_cal = data_holdout_full[evt_start_cal:evt_stop_cal, 1:]
data_masses_cal = data_holdout_full[evt_start_cal:evt_stop_cal, 0:1].reshape(-1)
labels_holdout_cal = labels_holdout_full[evt_start_cal:evt_stop_cal]

# template_bins = np.linspace(np.min(data_masses_cal), np.max(data_masses_cal), 50)


data_feats_cal = data_feats_cal[(data_masses_cal > cut_low) & (data_masses_cal < cut_high)]
labels_holdout_cal = labels_holdout_cal[(data_masses_cal > cut_low) & (data_masses_cal < cut_high)]
data_masses_cal = data_masses_cal[(data_masses_cal > cut_low) & (data_masses_cal < cut_high)]


# Filter signal to chosen amount
data_feats_cal = data_feats_cal[labels_holdout_cal==0]
data_masses_cal = data_masses_cal[labels_holdout_cal==0]
labels_holdout_cal = labels_holdout_cal[labels_holdout_cal==0]

SB_mask_cal = (data_masses_cal < mass_low_sr) | (data_masses_cal >= mass_high_sr)
SR_mask_cal = (data_masses_cal >= mass_low_sr) & (data_masses_cal < mass_high_sr)

# data_feats_cal = data_feats_cal[SB_mask_cal]
# data_masses_cal = data_masses_cal[SB_mask_cal]
# labels_holdout_cal = labels_holdout_cal[SB_mask_cal]

print("Calibration events", len(data_feats_cal))

        
bkg_flow_prob_cal = np.zeros(len(data_feats_cal))


overall_bkg_probs_cal = []

# separate preprocessing for signal and bkg flows
bkg_inputs_cal = bkg_scaler.transform(data_feats_cal)

# compute jacobian of preprocessing transformations
bkg_scaler_jacobian = np.ones((len(data_feats_cal), nfeatures))
for nfeat in range(nfeatures):
    h1 = (
        bkg_scaler["logitscaler"].scale_[nfeat] * data_feats_cal[:, nfeat]
        + bkg_scaler["logitscaler"].min_[nfeat]
    )
    h1 = np.where(
        h1 < 1 - bkg_scaler["logitscaler"].epsilon,
        h1,
        1 - bkg_scaler["logitscaler"].epsilon,
    )
    h1 = np.where(
        h1 > bkg_scaler["logitscaler"].epsilon, h1, bkg_scaler["logitscaler"].epsilon
    )

    bkg_scaler_jacobian[:, nfeat] = (
        (1 / bkg_scaler["standardscaler"].scale_[nfeat])
        * (1 / (h1 * (1 - h1)))
        * bkg_scaler["logitscaler"].scale_[nfeat]
    )

det_bkg_scaler_jacobian_cal = np.abs(np.prod(bkg_scaler_jacobian, 1))
print("Bkg jacobian")
print(np.sum(np.isnan(det_bkg_scaler_jacobian_cal)), np.sum(det_bkg_scaler_jacobian_cal == 0))
print(det_bkg_scaler_jacobian_cal[:10])



print("initial Nans")
print(np.sum(np.isnan(bkg_inputs_cal)))

# cleanup outliers, different procedure for high vs low nans
mean_data_feats_cal = np.mean(data_feats_cal, axis=0)
cutoff = 5.0
for i in range(data_feats_cal.shape[1]):
    above_avg = data_feats_cal[:, i] > mean_data_feats_cal[i]
    bkg_inputs_cal[:, i] = clean_nans(bkg_inputs_cal[:, i], above_avg)

bkg_inputs_cal = np.clip(bkg_inputs_cal, -cutoff, cutoff)


bkg_inputs_cal = torch.tensor(bkg_inputs_cal, dtype=torch.float32)    

bkg_flow_prob_cal += (
    flowB.log_prob(
        inputs=bkg_inputs_cal,
        context=torch.tensor(data_masses_cal.reshape(-1, 1), dtype=torch.float32),
    )
    .cpu()
    .detach()
    .numpy()
)    

bkg_flow_prob_cal = np.exp(bkg_flow_prob_cal) * det_bkg_scaler_jacobian_cal
if SB_flow_mask:
    bkg_flow_prob_cal[SB_mask_cal] = 1.0 / (2 * cutoff) ** 4



Get the mass parameters

In [ ]:
# nSig = 50
# data_iter = 2

# evt_start = data_iter * sample_size
# evt_stop = (data_iter + 1) * sample_size

# data_holdout_full = np.load(data_dir + "data_holdout.npy")
# labels_holdout_full = np.load(data_dir + "labels_holdout.npy")


# data_feats = data_holdout_full[evt_start:evt_stop, 1:]
# data_masses = data_holdout_full[evt_start:evt_stop, 0:1].reshape(-1)
# labels_holdout = labels_holdout_full[evt_start:evt_stop]

# template_bins = np.linspace(0.0, 1.0, 50)


# data_feats = data_feats[(data_masses > cut_low) & (data_masses < cut_high)]
# labels_holdout = labels_holdout[(data_masses > cut_low) & (data_masses < cut_high)]
# data_masses = data_masses[(data_masses > cut_low) & (data_masses < cut_high)]


# # Filter signal to chosen amount
# sig_mask = get_signal_mask(labels_holdout, nSig, seed=42)
# data_feats = data_feats[sig_mask]
# data_masses = data_masses[sig_mask]
# labels_holdout = labels_holdout[sig_mask]

# SB_mask = (data_masses < mass_low_sr) | (data_masses >= mass_high_sr)
# SR_mask = (data_masses >= mass_low_sr) & (data_masses < mass_high_sr)


fit_fn = ExpShape(np.min(data_masses_cal), np.max(data_masses_cal), order=3, num_integral=True)
# fit_fn = ExpShape(np.min(data_masses), np.max(data_masses), order=3, num_integral=True)
p0_guess = len(data_masses_cal)
p1_guess = 1.2
p2_guess = 0.1
p0_range = (0, None)
p1_range = (-100, 100)
p2_range = (-100, 100)

results_only_m_cal = analyser_no_signal(pdf_bkg_only_m, masses=data_masses_cal)
# results_only_m_cal = analyser_no_signal(pdf_bkg_only_m, masses=data_masses)

In [ ]:
results_only_m_cal['errors']

From which we infer the efficiencies

In [ ]:
all_prob_sim = bkg_flow_prob_sim * fit_fn.pdf(sim_masses_cal, results_only_m_cal['params'][1:]) 
all_prob_cal = bkg_flow_prob_cal * fit_fn.pdf(data_masses_cal, results_only_m_cal['params'][1:])

p0_fit, p1_fit, p2_fit = results_only_m_cal['params'][1:]
err_p0_fit, err_p1_fit, err_p2_fit = results_only_m_cal['errors'][1:]
print("Fit params", p0_fit, p1_fit, p2_fit)
print("Fit errors", err_p0_fit, err_p1_fit, err_p2_fit)
vals = np.linspace(np.min(data_masses_cal),np.max(data_masses_cal),100)

norm_toy = fit_fn.norm(results_only_m_cal['params'][1:])
Max_Val = 1.0*np.max(fit_fn.density(0.5*(vals[1:]+vals[:-1]), results_only_m_cal['params'][1:])/norm_toy)

# print("Normalization for toy", norm_toy)
data_masses_toys = np.zeros(len(data_masses_cal))
for i in range(len(data_masses_toys)):
    while True:
        mass_try = np.random.uniform(np.min(data_masses_cal), np.max(data_masses_cal))
        prob_try = fit_fn.density(mass_try, results_only_m_cal['params'][1:])/norm_toy
        if np.random.uniform(0,Max_Val) < prob_try:
            data_masses_toys[i] = mass_try
            break

plt.hist(data_masses_toys, bins=vals, density=True,histtype='step',label='Toy masses')
plt.plot(0.5*(vals[1:]+vals[:-1]), fit_fn.density(0.5*(vals[1:]+vals[:-1]), results_only_m_cal['params'][1:])/fit_fn.norm(results_only_m_cal['params'][1:]),label='Fit function',color='red')            
plt.show()



bkg_flow_prob_toys = np.zeros(len(data_masses_toys)) 

bkg_samples_toys = ensemble_sample(
        [flowB],
        context=torch.tensor(
            data_masses_toys.reshape(-1, 1), dtype=torch.float32
        ),
        scaler=bkg_scaler,
        num_samples=samples_per_model)


bkg_inputs_toys = bkg_scaler.transform(bkg_samples_toys)

# compute jacobian of preprocessing transformations
bkg_scaler_jacobian = np.ones((len(data_masses_toys), nfeatures))
for nfeat in range(nfeatures):
    h1 = (
        bkg_scaler["logitscaler"].scale_[nfeat] * bkg_samples_toys[:, nfeat]
        + bkg_scaler["logitscaler"].min_[nfeat]
    )
    h1 = np.where(
        h1 < 1 - bkg_scaler["logitscaler"].epsilon,
        h1,
        1 - bkg_scaler["logitscaler"].epsilon,
    )
    h1 = np.where(
        h1 > bkg_scaler["logitscaler"].epsilon, h1, bkg_scaler["logitscaler"].epsilon
    )

    bkg_scaler_jacobian[:, nfeat] = (
        (1 / bkg_scaler["standardscaler"].scale_[nfeat])
        * (1 / (h1 * (1 - h1)))
        * bkg_scaler["logitscaler"].scale_[nfeat]
    )

det_bkg_scaler_jacobian_toys = np.abs(np.prod(bkg_scaler_jacobian, 1))
# print("Bkg jacobian")
# print(np.sum(np.isnan(det_bkg_scaler_jacobian_toys)), np.sum(det_bkg_scaler_jacobian_toys == 0))
# print(det_bkg_scaler_jacobian_toys[:10])


# print("initial Nans")
# print(np.sum(np.isnan(bkg_inputs_toys)))

# cleanup outliers, different procedure for high vs low nans
mean_data_feats_toys = np.mean(bkg_samples_toys, axis=0)
cutoff = 5.0
for i in range(bkg_samples_toys.shape[1]):
    above_avg = bkg_samples_toys[:, i] > mean_data_feats_toys[i]
    bkg_inputs_toys[:, i] = clean_nans(bkg_inputs_toys[:, i], above_avg)

bkg_inputs_toys = np.clip(bkg_inputs_toys, -cutoff, cutoff)

bkg_inputs_toys = torch.tensor(bkg_inputs_toys, dtype=torch.float32)


bkg_flow_prob_toys += (
    flowB.log_prob(
        inputs=bkg_inputs_toys,
        context=torch.tensor(data_masses_toys.reshape(-1, 1), dtype=torch.float32),
    )
    .cpu()
    .detach()
    .numpy()
)

bkg_flow_prob_toys = np.exp(bkg_flow_prob_toys) * det_bkg_scaler_jacobian_toys

SB_mask_toys = (data_masses_toys < mass_low_sr) | (data_masses_toys >= mass_high_sr)
SR_mask_toys = (data_masses_toys >= mass_low_sr) & (data_masses_toys < mass_high_sr)


if SB_flow_mask:
    bkg_flow_prob_toys[SB_mask_toys] = 1.0 / (2 * cutoff) ** 4

all_prob_toys = bkg_flow_prob_toys * fit_fn.pdf(data_masses_toys, results_only_m_cal['params'][1:])

# plt.hist(all_prob_toys, alpha=0.5, label='Toys')
# plt.hist(all_prob_cal, alpha=0.5, label='Calibration Data')
# plt.yscale('log')
# plt.xlabel('Background Flow Probability Density')
# plt.ylabel('Events')
# plt.legend()
# # plt.savefig(plot_dir + "bkg_flow_prob_calibration.pdf")    
# plt.show()


npoints = 50
cuts = [np.quantile((all_prob_toys),i/npoints) for i in range(npoints+1)]    

fractions_toy, bins_toy = np.histogram(all_prob_toys,bins=cuts,density=False)
# print(fractions_toy.shape,likelihoods_toy.shape)
# print(np.sum(fractions_toy),len(likelihoods_toy))
fractions_toy=fractions_toy/np.sum(fractions_toy)
fractions, bins = np.histogram(all_prob_cal,bins=cuts,density=False)
fractions=fractions/np.sum(fractions)

vals = 0.5*(bins[1:]+bins[:-1])


accumulated_fractions_toy = np.array([np.sum(fractions_toy[:i+1]) for i in range(len(fractions_toy))])
accumulated_fractions = np.array([np.sum(fractions[:i+1]) for i in range(len(fractions_toy))])

eps = 1e-4

if np.max(np.abs(accumulated_fractions_toy-accumulated_fractions)) > eps:
    # Lmin = np.exp(-(0.5**2)/2)/(np.sqrt(2*np.pi)*np.std(XToy[yToy==0],ddof=1))#
    Lmin = vals[np.argmax(np.abs(accumulated_fractions_toy-accumulated_fractions))]
else:
    Lmin = 0.0
Lmin = 0.000178

Lmins_stored = Lmin

eff_toy = np.mean(all_prob_toys > Lmin)
eff_cal = np.mean(all_prob_cal > Lmin)
print("Efficiency at Lmin: Toy %.4f, Cal %.4f"%(eff_toy,eff_cal))

eff_sim = np.mean(all_prob_sim > Lmin)

eff_toy_stored = eff_toy
eff_cal_stored = eff_cal
eff_sim_stored = eff_sim

In [ ]:
# Lmin = 0.0005

In [ ]:
# Fit signal crystall ball shape
sig_masses = sim[:, 0:1]
sig_masses = sig_masses[:200000]
sig_masses_mask = (sig_masses > mass_low_sr + 0.01) & (sig_masses < mass_high_sr - 0.01)
sig_masses_fit = sig_masses[sig_masses_mask]
c = cost.UnbinnedNLL(sig_masses_fit, pdf_DCB)

plt.figure()
plt.hist(sig_masses_fit, bins=40)
# plt.savefig(results_dir + "sig_masses.pdf")
plt.show()


loc = np.mean(sig_masses_fit)
m_sig_dcb = Minuit(
    c,
    m_left=2.0,
    beta_left=2.0,
    scale_left=0.5,
    beta_right=2.0,
    m_right=2.0,
    scale_right=0.5,
    loc=loc,
)
m_sig_dcb.limits["scale_left"] = (0, None)
m_sig_dcb.limits["scale_right"] = (0, None)
m_sig_dcb.limits["beta_left"] = (1, None)
m_sig_dcb.limits["beta_right"] = (1, None)

m_sig_dcb.migrad()  # finds minimum
m_sig_dcb.hesse()  # accurately computes uncertainties

joblib.dump(m_sig_dcb.values[:], results_dir + "dcb_fit.pkl")


(
    beta_left_fit,
    m_left_fit,
    scale_left_fit,
    beta_right_fit,
    m_right_fit,
    scale_right_fit,
    loc_fit,
) = m_sig_dcb.values

sig_pdf = lambda x: len(sig_masses_fit) * cb.pdf(
    x,
    beta_left_fit,
    m_left_fit,
    scale_left_fit,
    beta_right_fit,
    m_right_fit,
    scale_right_fit,
    loc_fit,
)

# make_postfit_plot(
#     sig_masses_fit,
#     pdfs=[sig_pdf],
#     labels=["Signal"],
#     colors=["red"],
#     bins=50,
#     axis_label="Diphoton mass",
#     fname=results_dir + "dcb_fit.pdf",
# )

In [ ]:
print("Efficiency at Lmin %.6f: Toy %.4f, Cal %.4f, Sig %.4f"%(Lmin, eff_toy,eff_cal,eff_sim))

## Single run plots

In [ ]:
# print(np.std(XToy[yToy==0],ddof=1))
# other_Lmin = np.exp(-(1**2)/2)/(np.sqrt(2*np.pi)*np.std(XToy[yToy==0],ddof=1))#
# print('Other Lmin:',other_Lmin,np.sqrt(2*np.abs(np.log(np.sqrt(2*np.pi)*np.std(XToy[yToy==0],ddof=1)*other_Lmin))))
plt.scatter(vals,fractions_toy,label='Toy')        
plt.scatter(vals,fractions,label='True')        
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
# plt.axhline(np.max((accumulated_fractions_toy-accumulated_fractions)))#,label=r'$\mathcal{L}_{\text{min}}$')
plt.xlabel(r'$\mathcal{L}_{c}$')
plt.xscale('log')
# plt.ylabel(r'|Data-Toy| Accumulated fraction')
plt.ylabel(r'Hypersurface fraction of events')
plt.legend(loc='best')
plt.tight_layout()
# plt.savefig(f'{self.results_folder}/distribution_comparison.pdf')

plt.show() 


In [ ]:
print(Lmin)
plt.scatter(vals,np.abs(accumulated_fractions_toy-accumulated_fractions))        
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
plt.axhline(np.max(np.abs(accumulated_fractions_toy-accumulated_fractions)))#,label=r'$\mathcal{L}_{\text{min}}$')
plt.xlabel(r'$\mathcal{L}_{c}$')
plt.xscale('log')
# plt.ylabel(r'|Data-Toy| Accumulated fraction')
plt.ylabel(r'$|f(\mathcal{L}_{c}|\theta_{b}) - f(\mathcal{L}_{c}|\text{true b})|$')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(results_dir+'distribution_comparison.pdf')

plt.show()

In [ ]:
np.min(all_prob_toys),np.max(all_prob_toys),np.min(all_prob_cal),np.max(all_prob_cal)

In [ ]:
cuts_plots = cuts#[np.quantile((all_prob_toys),i/25) for i in range(25+1)]   
plt.hist(all_prob_toys,bins=cuts_plots, histtype='step',density=False, alpha=0.5, label='Toys')
plt.hist(all_prob_cal, bins=cuts_plots, histtype='step', density=False,alpha=0.5, label='Calibration Data')
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Background Flow Probability Density')
plt.ylabel('Events')
plt.legend()
plt.tight_layout()
plt.savefig(results_dir + "bkg_flow_prob_calibration_events.pdf")

In [ ]:
plt.hist(all_prob_toys,bins=cuts_plots, histtype='step',density=True, alpha=0.5, label='Toys')
plt.hist(all_prob_cal, bins=cuts_plots, histtype='step', density=True,alpha=0.5, label='Calibration Data')
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Background Flow Probability Density')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig(results_dir + "bkg_flow_prob_calibration.pdf")

A single run with 50 events

In [ ]:
nSig = 50
data_iter = 2

evt_start = data_iter * sample_size
evt_stop = (data_iter + 1) * sample_size

# data_holdout_full = np.load(data_dir + "data_holdout.npy")
# labels_holdout_full = np.load(data_dir + "labels_holdout.npy")


data_feats = data_holdout_full[evt_start:evt_stop, 1:]
data_masses = data_holdout_full[evt_start:evt_stop, 0:1].reshape(-1)
labels_holdout = labels_holdout_full[evt_start:evt_stop]

template_bins = np.linspace(0.0, 1.0, 50)


data_feats = data_feats[(data_masses > cut_low) & (data_masses < cut_high)]
labels_holdout = labels_holdout[(data_masses > cut_low) & (data_masses < cut_high)]
data_masses = data_masses[(data_masses > cut_low) & (data_masses < cut_high)]


# Filter signal to chosen amount
sig_mask = get_signal_mask(labels_holdout, nSig, seed=42)
data_feats = data_feats[sig_mask]
data_masses = data_masses[sig_mask]
labels_holdout = labels_holdout[sig_mask]

SB_mask = (data_masses < mass_low_sr) | (data_masses >= mass_high_sr)
SR_mask = (data_masses >= mass_low_sr) & (data_masses < mass_high_sr)

sig_eff_SR_mask = np.mean(SR_mask[labels_holdout.reshape(-1) > 0.5])


bkg_flow_prob = np.zeros(len(data_feats))
sig_flow_prob = np.zeros(len(data_feats))


overall_bkg_probs = []

# separate preprocessing for signal and bkg flows
bkg_inputs = bkg_scaler.transform(data_feats)

# compute jacobian of preprocessing transformations
bkg_scaler_jacobian = np.ones((len(data_feats), nfeatures))
for nfeat in range(nfeatures):
    h1 = (
        bkg_scaler["logitscaler"].scale_[nfeat] * data_feats[:, nfeat]
        + bkg_scaler["logitscaler"].min_[nfeat]
    )
    h1 = np.where(
        h1 < 1 - bkg_scaler["logitscaler"].epsilon,
        h1,
        1 - bkg_scaler["logitscaler"].epsilon,
    )
    h1 = np.where(
        h1 > bkg_scaler["logitscaler"].epsilon, h1, bkg_scaler["logitscaler"].epsilon
    )

    bkg_scaler_jacobian[:, nfeat] = (
        (1 / bkg_scaler["standardscaler"].scale_[nfeat])
        * (1 / (h1 * (1 - h1)))
        * bkg_scaler["logitscaler"].scale_[nfeat]
    )

det_bkg_scaler_jacobian = np.abs(np.prod(bkg_scaler_jacobian, 1))
print("Bkg jacobian")
print(np.sum(np.isnan(det_bkg_scaler_jacobian)), np.sum(det_bkg_scaler_jacobian == 0))
print(det_bkg_scaler_jacobian[:10])

sig_inputs = sig_scaler.transform(data_feats)

sig_scaler_jacobian = np.ones((len(data_feats), nfeatures))
for nfeat in range(nfeatures):
    h1 = (
        sig_scaler["logitscaler"].scale_[nfeat] * data_feats[:, nfeat]
        + sig_scaler["logitscaler"].min_[nfeat]
    )
    h1 = np.where(
        h1 < 1 - sig_scaler["logitscaler"].epsilon,
        h1,
        1 - sig_scaler["logitscaler"].epsilon,
    )
    h1 = np.where(
        h1 > sig_scaler["logitscaler"].epsilon, h1, sig_scaler["logitscaler"].epsilon
    )

    sig_scaler_jacobian[:, nfeat] = (
        (1 / sig_scaler["standardscaler"].scale_[nfeat])
        * (1 / (h1 * (1 - h1)))
        * sig_scaler["logitscaler"].scale_[nfeat]
    )

det_sig_scaler_jacobian = np.abs(np.prod(sig_scaler_jacobian, 1))

print("Sig jacobian")
print(np.sum(np.isnan(det_sig_scaler_jacobian)), np.sum(det_sig_scaler_jacobian == 0))
print(det_sig_scaler_jacobian[:10])

data_feats_sig = np.copy(data_feats)
# sideband events will be outside domain of sig flow training, so use central value
data_feats_sig[:, 0][SB_mask] = (mass_low_sr + mass_high_sr) / 2.0


print("initial Nans")
print(np.sum(np.isnan(bkg_inputs)), np.sum(np.isnan(sig_inputs)))

# cleanup outliers, different procedure for high vs low nans
mean_data_feats = np.mean(data_feats, axis=0)
cutoff = 5.0
for i in range(data_feats.shape[1]):
    above_avg = data_feats[:, i] > mean_data_feats[i]
    bkg_inputs[:, i] = clean_nans(bkg_inputs[:, i], above_avg)
    sig_inputs[:, i] = clean_nans(sig_inputs[:, i], above_avg)

bkg_inputs = np.clip(bkg_inputs, -cutoff, cutoff)
sig_inputs = np.clip(sig_inputs, -cutoff, cutoff)


bkg_inputs = torch.tensor(bkg_inputs, dtype=torch.float32)
sig_inputs = torch.tensor(sig_inputs, dtype=torch.float32)

In [ ]:
bkg_flow_prob += (
    flowB.log_prob(
        inputs=bkg_inputs,
        context=torch.tensor(data_masses.reshape(-1, 1), dtype=torch.float32),
    )
    .cpu()
    .detach()
    .numpy()
)

bkg_flow_prob = np.exp(bkg_flow_prob) * det_bkg_scaler_jacobian


In [ ]:
sig_flow_prob += (
    flowS.log_prob(
        inputs=sig_inputs,
        context=torch.tensor(data_masses.reshape(-1, 1), dtype=torch.float32),
    )
    .cpu()
    .detach()
    .numpy()
)

sig_flow_prob = np.exp(sig_flow_prob) * det_sig_scaler_jacobian


In [ ]:
likelihood_ratios = sig_flow_prob / bkg_flow_prob
s_over_b_ratios = sig_flow_prob / (sig_flow_prob + bkg_flow_prob) 

# ignore feature densities outside SR, where there is no relevant info
if SB_flow_mask:
    bkg_flow_prob[SB_mask] = 1.0 / (2 * cutoff) ** 4
    sig_flow_prob[SB_mask] = 1.0 / (2 * cutoff) ** 4


print(
    np.min(bkg_flow_prob),
    np.min(sig_flow_prob),
    np.max(bkg_flow_prob),
    np.max(sig_flow_prob),
)


vals_full = np.concatenate((data_masses.reshape(-1, 1), data_feats), axis=1)

In [ ]:
# remove nans
for i in range(data_feats.shape[1]):
    data_feats[:, i] = clean_nans(data_feats[:, i])
    
results_only_m = analyser(pdf_comb_only_m, masses=data_masses)
joblib.dump(results_only_m, results_dir + "m_only_fit.pkl")

# 30 points, +/- 3 sigma
num_grid = 250
mu_grid = np.linspace(
    results_only_m["best_s"] - results_only_m["best_s_err"],
    results_only_m["best_s"] + 2*results_only_m["best_s_err"],
    num_grid,
)



all_prob_flow = bkg_flow_prob * fit_fn.pdf(data_masses, results_only_m_cal['params'][1:])

# print("\n NF model")
# results_full = analyser(
#     pdf_comb_full,
#     masses=data_masses,
#     s_guess=results_only_m["best_s"],
#     # grid=mu_grid,
#     # plt_name=results_dir + "model_mass_fit.pdf",
# )



# # 30 points, +/- 3 sigma
# num_grid = 60
# mu_grid = np.linspace(
#     results_full["best_s"] - 3 * results_full["best_s_err"],
#     results_full["best_s"] + 3 * results_full["best_s_err"],
#     num_grid,
# )


# num_grid = 100
# mu_grid = np.linspace(
#     results_only_m["best_s"] -  results_only_m["best_s_err"],
#     results_full["best_s"] + results_full["best_s_err"],
#     num_grid,
# )

# joblib.dump(
#     results_full, results_dir + "model_fit.pkl" 
# )



Lmin = 0
eff_cal = 1.0
eff_sim = 1.0
eff_toy = 1.0
mask_mrl = all_prob_flow > Lmin

print("\n NF model")
results_full = analyser_full_Lmin(
    pdf_comb_full_Lmin,
    masses=data_masses[mask_mrl],
    s_guess=results_only_m["best_s"] ,
    verbose=False,
    # plt_name=plot_dir + "model_mrl_mass_fit.pdf",
)



# # 30 points, +/- 4 sigma
num_grid = 60
mu_grid = np.linspace(
    results_full["best_s"] - 3 * results_full["best_s_err"][0],
    results_full["best_s"] + 3 * results_full["best_s_err"][1],
    num_grid,
)



# joblib.dump(
#     results_full, results_dir + "model_fit.pkl" 
# )

efficiency = np.mean(all_prob_flow>Lmins_stored)
eff_cal = efficiency
eff_sim = efficiency
eff_toy = efficiency
efficiency_map = np.random.rand(len(all_prob_flow)) < efficiency
mask_mrl = efficiency_map

print("\n NF + lower stats model")
results_full_lower_stats = analyser_full_Lmin(
    pdf_comb_full_Lmin,
    masses=data_masses[mask_mrl],
    s_guess=results_only_m["best_s"] ,
    verbose=False,
    # plt_name=plot_dir + "model_mrl_mass_fit.pdf",
)



Lmin = Lmins_stored
eff_cal = eff_cal_stored
eff_sim = eff_sim_stored
eff_toy = eff_toy_stored
mask_mrl = all_prob_flow>Lmin

print("\n NF+MRL model")
results_full_mrl = analyser_full_Lmin(
    pdf_comb_full_Lmin,
    masses=data_masses[mask_mrl],
    s_guess=results_only_m["best_s"],
    verbose=False,
    # grid=mu_grid,
    # plt_name=plot_dir + "model_mrl_mass_fit.pdf",
)

mu_grid_mrl = np.linspace(
    results_full_mrl["best_s"] - 3 * results_full_mrl["best_s_err"][0],
    results_full_mrl["best_s"] + 3 * results_full_mrl["best_s_err"][1],
    num_grid,
)


joblib.dump(
    results_full_mrl, results_dir + "model_mrl_fit.pkl" 
)

# remove nans
for i in range(data_feats.shape[1]):
    data_feats[:, i] = clean_nans(data_feats[:, i])



In [ ]:
plt.hist(all_prob_toys,bins=cuts_plots, histtype='step',density=False, alpha=0.5, label='Toys')
plt.hist(all_prob_cal, bins=cuts_plots, histtype='step', density=False,alpha=0.5, label='Calibration Data')
plt.hist(all_prob_flow[labels_holdout==1], bins=cuts_plots, histtype='step', density=False,alpha=0.5, label='Signal Data')
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Background Flow Probability Density')
plt.ylabel('Events')
plt.legend()
plt.tight_layout()
# plt.savefig(results_dir + "bkg_flow_prob_calibration.pdf")
plt.show()

plt.hist(all_prob_toys,bins=cuts_plots, histtype='step',density=True, alpha=0.5, label='Toys')
plt.hist(all_prob_cal, bins=cuts_plots, histtype='step', density=True,alpha=0.5, label='Calibration Data')
plt.hist(all_prob_flow[labels_holdout==1], bins=cuts_plots, histtype='step', density=True,alpha=0.5, label='Signal Data')
plt.axvline(Lmin,label=r'$\mathcal{L}_{\text{min}}$')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Background Flow Probability Density')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
# plt.savefig(results_dir + "bkg_flow_prob_calibration.pdf")
plt.show()

In [ ]:
np.mean(efficiency_map)

In [ ]:
np.mean(all_prob_flow>Lmin)

In [ ]:
(results_full['params'][0]),(results_full_lower_stats['params'][0]),(results_full_mrl['params'][0])

In [ ]:
p0_fit,p1_fit,p2_fit

In [ ]:
results_full_mrl['params'][1:]

In [ ]:
plt.hist(data_masses, bins=np.linspace(0,1), density=True, histtype='step', label='Data')
xcenters = 0.5*(np.linspace(0,1)[1:] + np.linspace(0,1)[:-1])
plt.plot(xcenters,fit_fn.pdf(xcenters, results_only_m_cal['params'][1:]),label='Cal')
plt.plot(xcenters,fit_fn.pdf(xcenters, results_full_mrl['params'][2:]),label='SR')
plt.legend(loc='best')

In [ ]:
plt.hist(reverse_mgg(data_masses[labels_holdout<0.5]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background w/o cuts',density=False)
plt.hist(reverse_mgg(data_masses[labels_holdout>0.5]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal w/o cuts',density=False)
# plt.hist(reverse_mgg(data_masses[(labels_holdout<0.5) & (efficiency_map)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background after random cut',density=False)
# plt.hist(reverse_mgg(data_masses[(labels_holdout>0.5) & (efficiency_map)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal after random cut',density=False)
plt.hist(reverse_mgg(data_masses[(labels_holdout<0.5) & (all_prob_flow>Lmin)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background after MRL',density=False)
plt.hist(reverse_mgg(data_masses[(labels_holdout>0.5) & (all_prob_flow>Lmin)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal after MRL',density=False)
plt.xlabel(r'$m_{\gamma\gamma}$')
plt.ylabel('Events')
plt.legend(loc='center')
plt.title(f'$B ={round(np.sum(labels_holdout<0.5))}$, $S={round(np.sum(labels_holdout>0.5))}$')
plt.yscale('log')
plt.tight_layout()
plt.savefig(results_dir + "mass_distribution_after_MRL.pdf")
plt.show()


plt.hist(reverse_mgg(data_masses[labels_holdout<0.5]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background w/o cuts',density=True)
plt.hist(reverse_mgg(data_masses[labels_holdout>0.5]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal w/o cuts',density=True)
# plt.hist(reverse_mgg(data_masses[(labels_holdout<0.5) & (efficiency_map)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background after random cut',density=True)
# plt.hist(reverse_mgg(data_masses[(labels_holdout>0.5) & (efficiency_map)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal after random cut',density=True)
plt.hist(reverse_mgg(data_masses[(labels_holdout<0.5) & (all_prob_flow>Lmin)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Background after MRL',density=True)
plt.hist(reverse_mgg(data_masses[(labels_holdout>0.5) & (all_prob_flow>Lmin)]), bins=np.linspace(90, 180, 51), alpha=0.5, histtype='step', label='Signal after MRL',density=True)
plt.xlabel(r'$m_{\gamma\gamma}$')
plt.ylabel('Density')
plt.legend()
plt.title(f'$B ={round(np.sum(labels_holdout<0.5))}$, $S={round(np.sum(labels_holdout>0.5))}$')
plt.tight_layout()
plt.savefig(results_dir + "mass_density_after_MRL.pdf")

In [ ]:

NLL_explicit_profile = results_full["s_profile"]
significance_profile = 0


all_likelihoods_free = results_full["NLL_s_free"]
all_likelihoods_fixed = results_full["NLL_s_fixed"]

print("Likelihood scales")
print(all_likelihoods_free - np.min(all_likelihoods_free))
print(all_likelihoods_fixed - np.min(all_likelihoods_fixed))

deltaNLL_explicit = 2.0 * (np.min(all_likelihoods_fixed) - np.min(all_likelihoods_free))
# deltaNLL_explicit= np.min(all_likelihoods_fixed - all_likelihoods_free)
print(all_likelihoods_fixed)
print(all_likelihoods_free)
print(deltaNLL_explicit)
significance_profile = deltaNLL_explicit**0.5

explicit_profile_curve = 2 * (NLL_explicit_profile - np.min(NLL_explicit_profile))

# ymax = np.amax(explicit_profile_curve[ explicit_profile_curve < 30])
ymax = 20.0

true_nsig = np.sum(labels_holdout)

sigma_only_m = one_sig_CI(
    results_only_m["best_s"] / true_nsig,
    results_only_m["s_values"] / true_nsig,
    results_only_m["s_profile"],
)


eps = 1e-4
# Take mean of the different models as the best fit
best_s_explicit_profile = np.mean(mu_grid[NLL_explicit_profile < eps]) / true_nsig
sigma_explicit_profile = one_sig_CI(
    best_s_explicit_profile, mu_grid / true_nsig, NLL_explicit_profile
)

print("Model best fits", mu_grid[NLL_explicit_profile < eps] / true_nsig)

print("sigma_only_m", sigma_only_m, results_only_m["best_s_err"])

In [ ]:
results_full_mrl["s_values"][2*(results_full_mrl["s_profile"]-np.min(results_full_mrl["s_profile"])) < 1]/(true_nsig*eff_sim)

In [ ]:
true_nsig = np.sum(labels_holdout)

fig = plt.figure()
plt.plot(
    results_only_m["s_values"] / true_nsig,
    2 * (results_only_m["s_profile"] - np.min(results_only_m["s_profile"])),
    color="black",
    label=r"Mass fit : $%.2f_{-%.2f}^{+%.2f}$"
    % (results_only_m["best_s"] / true_nsig, sigma_only_m[0],sigma_only_m[1]),
    linestyle="solid",
)


# overall envelope
plt.plot(
    mu_grid / true_nsig,
    explicit_profile_curve,
    color="blue",
    label="HI-SIGMA : $%.2f_{-%.2f}^{+%.2f}$"
    % (
        best_s_explicit_profile,
        sigma_explicit_profile[0],
        sigma_explicit_profile[1],
    ),
    linestyle="solid",
)

# sigma_mrl = np.array(results_full_mrl["best_s_err"])/ (true_nsig)
sigma_lower_stats = one_sig_CI(
    results_full_lower_stats["best_s"] / (true_nsig),
    results_full_lower_stats["s_values"] / (true_nsig),
    results_full_lower_stats["s_profile"],
    results_full_lower_stats["best_s_err"][0]/ (true_nsig),
)

plt.plot(
    results_full_lower_stats["s_values"] / (true_nsig),
    2 * (results_full_lower_stats["s_profile"] - np.min(results_full_lower_stats["s_profile"])),
    color="goldenrod",
    label=r"HI-SIGMA + Random cut : $%.2f_{-%.2f}^{+%.2f} $"
    % (results_full_lower_stats["best_s"] / (true_nsig),sigma_lower_stats[0], sigma_lower_stats[1]),
    linestyle="solid",
)


# sigma_mrl = np.array(results_full_mrl["best_s_err"])/ (true_nsig)
sigma_mrl = one_sig_CI(
    results_full_mrl["best_s"] / (true_nsig),
    results_full_mrl["s_values"] / (true_nsig),
    results_full_mrl["s_profile"],
    results_full_mrl["best_s_err"][0]/ (true_nsig),
)

plt.plot(
    results_full_mrl["s_values"] / (true_nsig),
    2 * (results_full_mrl["s_profile"] - np.min(results_full_mrl["s_profile"])),
    color="darkgreen",
    label=r"HI-SIGMA + MRL : $%.2f_{-%.2f}^{+%.2f} $"
    % (results_full_mrl["best_s"] / (true_nsig),sigma_mrl[0], sigma_mrl[1]),
    linestyle="solid",
)

ax = plt.gca()
ax.tick_params(axis="both", which="major", labelsize=18)
plt.ylim([0, 9.1])
plt.xlabel("Signal Strength", fontsize=22)
plt.ylabel(r"- 2 $\Delta$ ln($\mathcal{L}$)", fontsize=22)
plt.axvline(1.0, color='darkred', linestyle='dashed', label='True signal strength')
plt.axhline(1.0,color='black', linestyle='dashed')
plt.axhline(4.0,color='black', linestyle='dashed')
plt.axhline(9.0,color='black', linestyle='dashed')
# plt.axvline(best_s_explicit_profile-sigma_explicit_profile[0],color='mediumblue')
# plt.axvline(best_s_explicit_profile+sigma_explicit_profile[1],color='mediumblue')
# plt.axvline(results_full_mrl["best_s"] / (true_nsig)-sigma_mrl[0],color='darkgreen')
# plt.axvline(results_full_mrl["best_s"] / (true_nsig)+sigma_mrl[1],color='darkgreen')
plt.legend(loc="upper center")
plt.title(f'$B ={round(np.sum(labels_holdout<0.5))}$, $S={round(np.sum(labels_holdout>0.5))}$')
plt.tight_layout()
plt.savefig(results_dir + "likelihood_cmp_with_mrl.pdf")

## Coverage test

In [ ]:
# nSigs = np.array([10,25,50,100,150,200])
nSigs = np.array([10.0,25.0,75.0,100.0,150.0,300.0])
# nSigs = np.logspace(np.log10(50),np.log10(200),10)
print(nSigs)
# nSigs = np.array([100,200])
central_values = np.zeros((len(nSigs),data_iter_max,3))
errors = np.zeros((len(nSigs),data_iter_max,3))
coverage = np.zeros((len(nSigs),data_iter_max,3))
Lmins = np.zeros(data_iter_max)

In [ ]:
for data_iter in range(data_iter_max-1):
    evt_start = data_iter * sample_size
    evt_stop = (data_iter + 1) * sample_size

    # if data_iter < data_iter_max - 1:
    #     evt_start_cal = (data_iter + 1) * sample_size
    #     evt_stop_cal = (data_iter + 2) * sample_size
    # else:
    #     evt_start_cal = 0
    #     evt_stop_cal = sample_size

    for nnSig, nSig in enumerate(nSigs):
        plots_dir = results_dir + "plots_nSig%d/data_iter%d/"%(nSig,data_iter)
        os.makedirs(plots_dir, exist_ok=True)
        data_feats = data_holdout_full[evt_start:evt_stop, 1:]
        data_masses = data_holdout_full[evt_start:evt_stop, 0:1].reshape(-1)
        labels_holdout = labels_holdout_full[evt_start:evt_stop]

        template_bins = np.linspace(0,1, 50)

        data_feats = data_feats[(data_masses > cut_low) & (data_masses < cut_high)]
        labels_holdout = labels_holdout[(data_masses > cut_low) & (data_masses < cut_high)]
        data_masses = data_masses[(data_masses > cut_low) & (data_masses < cut_high)]


        # Filter signal to chosen amount
        print(nnSig,nSig)
        sig_mask = get_signal_mask(labels_holdout, nSig, seed=42)
        data_feats = data_feats[sig_mask]
        data_masses = data_masses[sig_mask]
        labels_holdout = labels_holdout[sig_mask]

        true_nsig = np.sum(labels_holdout)
        print(true_nsig,np.sum(labels_holdout==0))

        SB_mask = (data_masses < mass_low_sr) | (data_masses >= mass_high_sr)
        SR_mask = (data_masses >= mass_low_sr) & (data_masses < mass_high_sr)

        sig_eff_SR_mask = np.mean(SR_mask[labels_holdout.reshape(-1) > 0.5])


        bkg_flow_prob = np.zeros(len(data_feats))
        sig_flow_prob = np.zeros(len(data_feats))


        overall_bkg_probs = []

        # separate preprocessing for signal and bkg flows
        bkg_inputs = bkg_scaler.transform(data_feats)

        # compute jacobian of preprocessing transformations
        bkg_scaler_jacobian = np.ones((len(data_feats), nfeatures))
        for nfeat in range(nfeatures):
            h1 = (
                bkg_scaler["logitscaler"].scale_[nfeat] * data_feats[:, nfeat]
                + bkg_scaler["logitscaler"].min_[nfeat]
            )
            h1 = np.where(
                h1 < 1 - bkg_scaler["logitscaler"].epsilon,
                h1,
                1 - bkg_scaler["logitscaler"].epsilon,
            )
            h1 = np.where(
                h1 > bkg_scaler["logitscaler"].epsilon, h1, bkg_scaler["logitscaler"].epsilon
            )

            bkg_scaler_jacobian[:, nfeat] = (
                (1 / bkg_scaler["standardscaler"].scale_[nfeat])
                * (1 / (h1 * (1 - h1)))
                * bkg_scaler["logitscaler"].scale_[nfeat]
            )

        det_bkg_scaler_jacobian = np.abs(np.prod(bkg_scaler_jacobian, 1))
        # print("Bkg jacobian")
        # print(np.sum(np.isnan(det_bkg_scaler_jacobian)), np.sum(det_bkg_scaler_jacobian == 0))
        # print(det_bkg_scaler_jacobian[:10])

        sig_inputs = sig_scaler.transform(data_feats)

        sig_scaler_jacobian = np.ones((len(data_feats), nfeatures))
        for nfeat in range(nfeatures):
            h1 = (
                sig_scaler["logitscaler"].scale_[nfeat] * data_feats[:, nfeat]
                + sig_scaler["logitscaler"].min_[nfeat]
            )
            h1 = np.where(
                h1 < 1 - sig_scaler["logitscaler"].epsilon,
                h1,
                1 - sig_scaler["logitscaler"].epsilon,
            )
            h1 = np.where(
                h1 > sig_scaler["logitscaler"].epsilon, h1, sig_scaler["logitscaler"].epsilon
            )

            sig_scaler_jacobian[:, nfeat] = (
                (1 / sig_scaler["standardscaler"].scale_[nfeat])
                * (1 / (h1 * (1 - h1)))
                * sig_scaler["logitscaler"].scale_[nfeat]
            )

        det_sig_scaler_jacobian = np.abs(np.prod(sig_scaler_jacobian, 1))

        # print("Sig jacobian")
        # print(np.sum(np.isnan(det_sig_scaler_jacobian)), np.sum(det_sig_scaler_jacobian == 0))
        # print(det_sig_scaler_jacobian[:10])

        data_feats_sig = np.copy(data_feats)
        # sideband events will be outside domain of sig flow training, so use central value
        data_feats_sig[:, 0][SB_mask] = (mass_low_sr + mass_high_sr) / 2.0


        # print("initial Nans")
        # print(np.sum(np.isnan(bkg_inputs)), np.sum(np.isnan(sig_inputs)))

        # cleanup outliers, different procedure for high vs low nans
        mean_data_feats = np.mean(data_feats, axis=0)
        cutoff = 5.0
        for i in range(data_feats.shape[1]):
            above_avg = data_feats[:, i] > mean_data_feats[i]
            bkg_inputs[:, i] = clean_nans(bkg_inputs[:, i], above_avg)
            sig_inputs[:, i] = clean_nans(sig_inputs[:, i], above_avg)

        bkg_inputs = np.clip(bkg_inputs, -cutoff, cutoff)
        sig_inputs = np.clip(sig_inputs, -cutoff, cutoff)


        bkg_inputs = torch.tensor(bkg_inputs, dtype=torch.float32)
        sig_inputs = torch.tensor(sig_inputs, dtype=torch.float32)  

        bkg_flow_prob += (
            flowB.log_prob(
                inputs=bkg_inputs,
                context=torch.tensor(data_masses.reshape(-1, 1), dtype=torch.float32),
            )
            .cpu()
            .detach()
            .numpy()
        )

        bkg_flow_prob = np.exp(bkg_flow_prob) * det_bkg_scaler_jacobian

        sig_flow_prob += (
            flowS.log_prob(
                inputs=sig_inputs,
                context=torch.tensor(data_masses.reshape(-1, 1), dtype=torch.float32),
            )
            .cpu()
            .detach()
            .numpy()
        )
        sig_flow_prob = np.exp(sig_flow_prob) * det_sig_scaler_jacobian


        # ignore feature densities outside SR, where there is no relevant info
        if SB_flow_mask:
            bkg_flow_prob[SB_mask] = 1.0 / (2 * cutoff) ** 4
            sig_flow_prob[SB_mask] = 1.0 / (2 * cutoff) ** 4  

        
        all_prob_flow = bkg_flow_prob * fit_fn.pdf(data_masses, results_only_m_cal['params'][1:])

        results_only_m = analyser(pdf_comb_only_m, masses=data_masses)
        # joblib.dump(results_only_m, plot_dir + "m_only_fit.pkl")
        central_values[nnSig,data_iter,0]=results_only_m['best_s']
        sigma_only_m = one_sig_CI(
            results_only_m["best_s"] ,
            results_only_m["s_values"],
            results_only_m["s_profile"],
        )        
        errors[nnSig,data_iter,0]=0.5*(sigma_only_m[0]+sigma_only_m[1])
        if np.abs(central_values[nnSig,data_iter,0]-true_nsig) < errors[nnSig,data_iter,0]:
            coverage[nnSig,data_iter,0] = 1


        

        # results_full = analyser(
        #     pdf_comb_full,
        #     masses=data_masses,
        #     s_guess=results_only_m["best_s"]
        # )
        # sigma_full = one_sig_CI(
        #     results_full["best_s"] ,
        #     results_full["s_values"],
        #     results_full["s_profile"],
        # )        
        # central_values[nnSig,data_iter,1]=results_full['best_s']
        # errors[nnSig,data_iter,1]=0.5*(sigma_full[0]+sigma_full[1])
        # if np.abs(central_values[nnSig,data_iter,1]-true_nsig) < errors[nnSig,data_iter,1]:
        #     coverage[nnSig,data_iter,1] = 1
    
        # if np.min(results_full["s_values"])>0:
        #     # grid = np.linspace(np.min(results_full["s_values"]),1.2*np.max(results_full["s_values"]),1000)*eff_cal
        #     grid = np.linspace(results_full["best_s"] - 2 * sigma_full[0],results_full["best_s"] + 2 * sigma_full[1],1000) * eff_cal
        # else:
        #     # grid = np.linspace(1.2*np.min(results_full["s_values"]),1.8*np.max(results_full["s_values"]),1000)*eff_cal
        #     grid = np.linspace(results_full["best_s"] - 2 * sigma_full[0],results_full["best_s"] + 2 * sigma_full[1],1000) * eff_cal

        Lmin = 0.0
        eff_toy = 1
        eff_cal = 1
        eff_sim = 1
        mask_mrl = all_prob_flow>Lmin        
        print(np.sum(all_prob_flow>Lmin),np.sum(labels_holdout[all_prob_flow>Lmin]))
        results_full = analyser_full_Lmin(
            pdf_comb_full_Lmin,
            masses=data_masses[mask_mrl],
            s_guess=results_only_m["best_s"],
            verbose=False,
            # grid = np.linspace(np.min(results_only_m["s_values"]),np.max(results_only_m["s_values"]),1000)*eff_cal,
            # grid = grid,
            # plt_name=plot_dir + "model_mrl_mass_fit.pdf",
        )

        sigma_full = one_sig_CI(
            results_full["best_s"],
            results_full["s_values"] ,
            results_full["s_profile"],
            results_full["best_s_err"][0],
        )
        # joblib.dump(results_only_m, plot_dir + "m_only_fit.pkl")
        central_values[nnSig,data_iter,1]=results_full['best_s']
        errors[nnSig,data_iter,1]=0.5*(sigma_full[0]+sigma_full[1])
        if np.abs(central_values[nnSig,data_iter,1]-true_nsig) < errors[nnSig,data_iter,1]:
            coverage[nnSig,data_iter,1] = 1     

        Lmin = Lmins_stored
        eff_toy = eff_toy_stored
        eff_cal = eff_cal_stored
        eff_sim = eff_sim_stored
        mask_mrl = all_prob_flow>Lmin
        print(np.sum(all_prob_flow>Lmin),np.sum(labels_holdout[all_prob_flow>Lmin]))
        results_full_mrl = analyser_full_Lmin(
            pdf_comb_full_Lmin,
            masses=data_masses[mask_mrl],
            s_guess=results_only_m["best_s"],
            verbose=False,
            # grid = np.linspace(np.min(results_only_m["s_values"]),np.max(results_only_m["s_values"]),1000)*eff_cal,
            # grid = grid,
            # plt_name=plot_dir + "model_mrl_mass_fit.pdf",
        )

        sigma_mrl = one_sig_CI(
            results_full_mrl["best_s"],
            results_full_mrl["s_values"] ,
            results_full_mrl["s_profile"],
            results_full_mrl["best_s_err"][0],
        )
        # joblib.dump(results_only_m, plot_dir + "m_only_fit.pkl")
        central_values[nnSig,data_iter,2]=results_full_mrl['best_s']
        errors[nnSig,data_iter,2]=0.5*(sigma_mrl[0]+sigma_mrl[1])
        if np.abs(central_values[nnSig,data_iter,2]-true_nsig) < errors[nnSig,data_iter,2]:
            coverage[nnSig,data_iter,2] = 1     

        fig = plt.figure()
        plt.plot(
            results_only_m["s_values"] / true_nsig,
            2 * (results_only_m["s_profile"] - np.min(results_only_m["s_profile"])),
            color="black",
            label=r"Mass fit : $%.2f_{-%.2f}^{+%.2f}$"
            % (results_only_m["best_s"] / true_nsig, sigma_only_m[0]/ true_nsig,sigma_only_m[1]/ true_nsig),
            linestyle="solid",
        )


        # overall envelope
        plt.plot(
            results_full["s_values"] / (true_nsig),
            2 * (results_full["s_profile"] - np.min(results_full["s_profile"])),
            color="mediumblue",
            label=r"HI-SIGMA : $%.2f_{-%.2f}^{+%.2f} $"
            % (results_full["best_s"] / (true_nsig),sigma_full[0]/ true_nsig, sigma_full[1]/ true_nsig),
            linestyle="solid",
        )



        plt.plot(
            results_full_mrl["s_values"] / (true_nsig),
            2 * (results_full_mrl["s_profile"] - np.min(results_full_mrl["s_profile"])),
            color="darkgreen",
            label=r"HI-SIGMA + MRL : $%.2f_{-%.2f}^{+%.2f} $"
            % (results_full_mrl["best_s"] / (true_nsig),sigma_mrl[0]/ true_nsig, sigma_mrl[1]/ true_nsig),
            linestyle="solid",
        )

        ax = plt.gca()
        ax.tick_params(axis="both", which="major", labelsize=18)
        plt.ylim([0, 9])
        plt.xlabel("Signal Strength", fontsize=22)
        plt.ylabel(r"- 2 $\Delta$ ln($\mathcal{L}$)", fontsize=22)
        plt.axvline(1.0, color='red', linestyle='dashed', label='True signal strength')
        plt.axhline(1.0,color='red')
        plt.axvline(results_full["best_s"] / (true_nsig)-sigma_full[0]/ true_nsig,color='mediumblue')
        plt.axvline(results_full["best_s"] / (true_nsig)+sigma_full[1]/ true_nsig,color='mediumblue')
        plt.axvline(results_full_mrl["best_s"] / (true_nsig)-sigma_mrl[0]/ true_nsig,color='darkgreen')
        plt.axvline(results_full_mrl["best_s"] / (true_nsig)+sigma_mrl[1]/ true_nsig,color='darkgreen')
        plt.legend(loc="upper center")
        plt.title(f'$B ={round(np.sum(labels_holdout<0.5))}$, $S={round(np.sum(labels_holdout>0.5))}$')
        plt.tight_layout()
        plt.savefig(plots_dir+'fit_plot.pdf')
        plt.savefig(plots_dir+'fit_plot.png')
        plt.show()

        

In [ ]:
# central_values[:,:-1,1],
errors[:,:-1,2]

In [ ]:
np.save(results_dir + "central_values.npy", central_values)
np.save(results_dir + "errors.npy", errors)
np.save(results_dir + "coverage.npy", coverage)

In [ ]:
central_values = np.load(results_dir + "central_values.npy")
errors = np.load(results_dir + "errors.npy")
coverage = np.load(results_dir + "coverage.npy")

In [ ]:
import scipy.stats as st
for nnSig, nSig in enumerate(nSigs):
    if nnSig <= -1:
        pass
    elif nnSig == 0:
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,0]/nSig, yerr=errors[nnSig,:-1,0]/nSig, fmt='ko', alpha=0.5,label='Mass fit')
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,1]/nSig, yerr=errors[nnSig,:-1,1]/nSig, color='mediumblue', fmt='x', alpha=0.5,label='HI-SIGMA')
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,2]/nSig, yerr=errors[nnSig,:-1,2]/nSig, color='darkgreen',fmt='+', alpha=0.5,label='HI-SIGMA + MRL')
    else:
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,0]/nSig, yerr=errors[nnSig,:-1,0]/nSig, fmt='bo', alpha=0.5)
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,1]/nSig, yerr=errors[nnSig,:-1,1]/nSig , color='mediumblue', fmt='x', alpha=0.5)
        plt.errorbar(nSig+st.norm(0,1).rvs(data_iter_max-1),central_values[nnSig,:-1,2]/nSig, yerr=errors[nnSig,:-1,2]/nSig,color='darkgreen', fmt='+', alpha=0.5)
    # print(neffs)
    # print(np.mean(neffs),np.std(neffs,ddof=1))
    # plt.text(nSig, 1.2, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,1]/nSig), np.mean(errors[nnSig,:-1,1]/nSig)), ha='center', va='bottom', fontsize=10, color='red',bbox=dict(facecolor='white', alpha=0.5) )
    # plt.text(nSig, 0.8, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,2]/nSig), np.mean(errors[nnSig,:-1,2]/nSig)), ha='center', va='bottom', fontsize=10, color='green',bbox=dict(facecolor='white', alpha=0.5) )
    if nnSig <= -1:
        pass
    else:
        plt.annotate(
            "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,0]/nSig), np.mean(errors[nnSig,:-1,0]/nSig)),
            xy=(nSig, 1.0),
            xytext=(nSig, 4+3*(2*(len(nSigs)-0-nnSig)/(len(nSigs)-0))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='black',bbox=dict(facecolor='white', alpha=0.75) 
        )    


        plt.annotate(
            "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,1]/nSig), np.mean(errors[nnSig,:-1,1]/nSig)),
            xy=(nSig, 1.0),
            xytext=(nSig, 2+3*(2*(len(nSigs)-0-nnSig)/(len(nSigs)-0))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='mediumblue',bbox=dict(facecolor='white', alpha=0.75) 
        )    
        plt.annotate(
            "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,2]/nSig), np.mean(errors[nnSig,:-1,2]/nSig)),
            xy=(nSig, 1.0),
            xytext=(nSig, -2-3*(2*(len(nSigs)-0-nnSig)/(len(nSigs)-0))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='darkgreen',bbox=dict(facecolor='white', alpha=0.75) 
        )
plt.axhline(1,color='k',ls='--',alpha=0.25)

plt.xticks(nSigs[1:])
plt.xlabel('True signal events')
plt.ylabel('Fitted signal strength')
# plt.xscale('log')
# plt.yscale('log')
plt.legend(loc='best')
# plt.savefig(plot_dir+'signal_injection_study.png')
plt.tight_layout()
plt.savefig(results_dir+'signal_injection_study_signal_strength.pdf')
plt.savefig(results_dir+'signal_injection_study_signal_strength.png')

In [ ]:
coverage.shape

In [ ]:
for nnSig, nSig in enumerate(nSigs):
    if nnSig <= -1:
        pass
    elif nnSig == 0:
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,0]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,0]))*(1-np.mean(coverage[nnSig,:-1,0]))/len(central_values[nnSig,:-1])), color='black', fmt='o', alpha=0.5,label='Mass fit')
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,1]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,1]))*(1-np.mean(coverage[nnSig,:-1,1]))/len(central_values[nnSig,:-1])), color='mediumblue', fmt='x', alpha=0.5,label='HI-SIGMA')
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,2]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,2]))*(1-np.mean(coverage[nnSig,:-1,2]))/len(central_values[nnSig,:-1])), color='darkgreen',fmt='+', alpha=0.5,label='HI-SIGMA + MRL')
    else:
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,0]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,0]))*(1-np.mean(coverage[nnSig,:-1,0]))/len(central_values[nnSig,:-1])), color='black', fmt='o', alpha=0.5)
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,1]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,1]))*(1-np.mean(coverage[nnSig,:-1,1]))/len(central_values[nnSig,:-1])), color='mediumblue', fmt='x', alpha=0.5)
        plt.errorbar(nSig,np.mean(coverage[nnSig,:-1,2]),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,2]))*(1-np.mean(coverage[nnSig,:-1,2]))/len(central_values[nnSig,:-1])), color='darkgreen',fmt='+', alpha=0.5)
    # print(neffs)
    # print(np.mean(neffs),np.std(neffs,ddof=1))
    # plt.text(nSig, 1.2, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,1]/nSig), np.mean(errors[nnSig,:-1,1]/nSig)), ha='center', va='bottom', fontsize=10, color='red',bbox=dict(facecolor='white', alpha=0.5) )
    # plt.text(nSig, 0.8, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,2]/nSig), np.mean(errors[nnSig,:-1,2]/nSig)), ha='center', va='bottom', fontsize=10, color='green',bbox=dict(facecolor='white', alpha=0.5) )
    if nnSig <= -1:
        pass
    else:    
        plt.annotate(
            "$p=%8.2f \pm %8.2f$" % (np.mean(coverage[nnSig,:-1,0]),np.sqrt((np.mean(coverage[nnSig,:-1,0]))*(1-np.mean(coverage[nnSig,:-1,0]))/len(central_values[nnSig,:-1]))),
            xy=(nSig, 0.68),
            # xytext=(nSig,(1.0/5)*(2+3*(2*(len(nSigs)-nnSig)/len(nSigs))**1)),
             xytext=(nSig,0.9-0.25*((nnSig)/len(nSigs))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='black',bbox=dict(facecolor='white', alpha=0.75) 
        )   
        plt.annotate(
            "$p=%8.2f \pm %8.2f$" % (np.mean(coverage[nnSig,:-1,1]),np.sqrt((np.mean(coverage[nnSig,:-1,1]))*(1-np.mean(coverage[nnSig,:-1,1]))/len(central_values[nnSig,:-1]))),
            xy=(nSig, 0.68),
            # xytext=(nSig,(0.3/5)*(2+3*(2*(len(nSigs)-nnSig)/len(nSigs))**1)),
            xytext=(nSig,0.3-0.25*((nnSig)/len(nSigs))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='mediumblue',bbox=dict(facecolor='white', alpha=0.75) 
        )    
        plt.annotate(
            "$p=%8.2f \pm %8.2f$" % (np.mean(coverage[nnSig,:-1,2]),np.sqrt((np.mean(coverage[nnSig,:-1,2]))*(1-np.mean(coverage[nnSig,:-1,2]))/len(central_values[nnSig,:-1]))),
            xy=(nSig, 0.68),
            # xytext=(nSig,  (0.8/5)*(2+3*(2*(len(nSigs)-nnSig)/len(nSigs))**1)),
            xytext=(nSig,  0.8-0.25*((nnSig)/len(nSigs))**1),
            # arrowprops=dict(facecolor="black", shrink=0.05),
            horizontalalignment="center",
            fontsize=10,
            color='darkgreen',bbox=dict(facecolor='white', alpha=0.75) 
        )
    plt.axhline(0.68,color='k',ls='--',alpha=0.25)

    plt.xticks(nSigs[1:])
    plt.ylim(0,1)
    plt.xlabel('True signal events')
    plt.ylabel('Coverage test')
    # plt.xscale('log')
    # plt.yscale('log')
    plt.legend(loc='best')
    # plt.savefig(plot_dir+'signal_injection_study.png')
    plt.tight_layout()
    plt.savefig(results_dir+'signal_injection_study_signal_strength_coverage.pdf')
    plt.savefig(results_dir+'signal_injection_study_signal_strength_coverage.png')

In [ ]:
np.mean(central_values[:,:-1,1]<0,1),np.mean(coverage[:,:-1,1],1)

In [ ]:
np.mean(central_values[:,:-1,2]<0,1),np.mean(coverage[:,:-1,2],1)

In [ ]:
for nnSig, nSig in enumerate(nSigs):
    if nnSig <= -1:
        pass
    elif nnSig == 0:
        # plt.errorbar(np.mean(central_values[nnSig,:-1,0]<0),np.mean(coverage[nnSig,:-1,0]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,0]<0))*(1-np.mean(central_values[nnSig,:-1,0]<0))/len(central_values[nnSig,:-1,1])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,0]))*(1-np.mean(coverage[nnSig,:-1,0]))/len(central_values[nnSig,:-1])), color='black', fmt='o', alpha=0.5,label='Mass fit')
        plt.errorbar(np.mean(central_values[nnSig,:-1,1]<0),np.mean(coverage[nnSig,:-1,1]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,1]<0))*(1-np.mean(central_values[nnSig,:-1,1]<0))/len(central_values[nnSig,:-1,1])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,1]))*(1-np.mean(coverage[nnSig,:-1,1]))/len(central_values[nnSig,:-1])), color='mediumblue', fmt='x', alpha=0.5,label='HI-SIGMA')
        plt.errorbar(np.mean(central_values[nnSig,:-1,2]<0),np.mean(coverage[nnSig,:-1,2]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,2]<0))*(1-np.mean(central_values[nnSig,:-1,2]<0))/len(central_values[nnSig,:-1,2])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,2]))*(1-np.mean(coverage[nnSig,:-1,2]))/len(central_values[nnSig,:-1])), color='darkgreen',fmt='+', alpha=0.5,label='HI-SIGMA + MRL')
    else:
        # plt.errorbar(np.mean(central_values[nnSig,:-1,0]<0),np.mean(coverage[nnSig,:-1,0]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,0]<0))*(1-np.mean(central_values[nnSig,:-1,0]<0))/len(central_values[nnSig,:-1,1])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,0]))*(1-np.mean(coverage[nnSig,:-1,0]))/len(central_values[nnSig,:-1])), color='black', fmt='o', alpha=0.5)
        plt.errorbar(np.mean(central_values[nnSig,:-1,1]<0),np.mean(coverage[nnSig,:-1,1]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,1]<0))*(1-np.mean(central_values[nnSig,:-1,1]<0))/len(central_values[nnSig,:-1,1])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,1]))*(1-np.mean(coverage[nnSig,:-1,1]))/len(central_values[nnSig,:-1])), color='mediumblue', fmt='x', alpha=0.5)
        plt.errorbar(np.mean(central_values[nnSig,:-1,2]<0),np.mean(coverage[nnSig,:-1,2]),xerr=np.sqrt((np.mean(central_values[nnSig,:-1,2]<0))*(1-np.mean(central_values[nnSig,:-1,2]<0))/len(central_values[nnSig,:-1,2])),yerr=np.sqrt((np.mean(coverage[nnSig,:-1,2]))*(1-np.mean(coverage[nnSig,:-1,2]))/len(central_values[nnSig,:-1])), color='darkgreen',fmt='+', alpha=0.5)
    # print(neffs)
    # print(np.mean(neffs),np.std(neffs,ddof=1))
    # plt.text(nSig, 1.2, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,1]/nSig), np.mean(errors[nnSig,:-1,1]/nSig)), ha='center', va='bottom', fontsize=10, color='red',bbox=dict(facecolor='white', alpha=0.5) )
    # plt.text(nSig, 0.8, "$\mu=%8.2f \pm %8.2f$" % (np.mean(central_values[nnSig,:-1,2]/nSig), np.mean(errors[nnSig,:-1,2]/nSig)), ha='center', va='bottom', fontsize=10, color='green',bbox=dict(facecolor='white', alpha=0.5) )
    # plt.annotate(
    #     "$p=%8.2f \pm %8.2f$" % (np.mean(np.abs(central_values[nnSig,:-1,1]-nSig)<errors[nnSig,:-1,1]),np.sqrt((np.mean(np.abs(central_values[nnSig,:-1,1]-nSig)<errors[nnSig,:-1,1]))*(1-np.mean(np.abs(central_values[nnSig,:-1,1]-nSig)<errors[nnSig,:-1,1]))/len(central_values[nnSig,:-1]))),
    #     xy=(nSig, 0.68),
    #     xytext=(nSig,(0.3/5)*(2+3*((len(nSigs)-nnSig)/len(nSigs))**2)),
    #     # arrowprops=dict(facecolor="black", shrink=0.05),
    #     horizontalalignment="center",
    #     fontsize=10,
    #     color='mediumblue',bbox=dict(facecolor='white', alpha=0.75) 
    # )    
    # plt.annotate(
    #     "$p=%8.2f \pm %8.2f$" % (np.mean(np.abs(central_values[nnSig,:-1,2]-nSig)<errors[nnSig,:-1,2]),np.sqrt((np.mean(np.abs(central_values[nnSig,:-1,2]-nSig)<errors[nnSig,:-1,2]))*(1-np.mean(np.abs(central_values[nnSig,:-1,2]-nSig)<errors[nnSig,:-1,2]))/len(central_values[nnSig,:-1]))),
    #     xy=(nSig, 0.68),
    #     xytext=(nSig,  (0.8/5)*(2+3*((len(nSigs)-nnSig)/len(nSigs))**2)),
    #     # arrowprops=dict(facecolor="black", shrink=0.05),
    #     horizontalalignment="center",
    #     fontsize=10,
    #     color='darkgreen',bbox=dict(facecolor='white', alpha=0.75) 
    # )
plt.axhline(0.68,color='k',ls='--',alpha=0.25)
plt.axvline(0.0,color='k',ls='--',alpha=0.25)

# plt.xticks(nSigs[1:])
plt.xlabel('Fraction of negative $\hat{S}$')
plt.ylabel('Coverage test')
# plt.xscale('log')
# plt.yscale('log')
plt.legend(loc='best')
# plt.savefig(plot_dir+'signal_injection_study.png')
plt.tight_layout()
plt.savefig(results_dir+'coverage_negative_fraction.pdf')
plt.savefig(results_dir+'coverage_negative_fraction.png')